In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import pickle
import copy

from imports import *
from config import dir_config, ephys_config
from src.utils import dpca_utils, dpca_plot_utils

In [ ]:
compiled_dir  = Path(dir_config.data.compiled)
processed_dir = Path(dir_config.data.processed)

## Load data

Data loading stays in the notebook. The utils start after you have
`session_metadata`, `neuron_metadata`, `ephys`, and the trial-info dicts.

In [ ]:
session_to_exclude = ["210210_GP_JP", "241209_GP_TZ"]

session_metadata = pd.read_csv(Path(processed_dir, "sessions_metadata.csv"))
session_metadata = session_metadata[~session_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

neuron_metadata = pd.read_csv(Path(processed_dir, "neuron_metadata.csv"))
neuron_metadata = neuron_metadata[~neuron_metadata.session_id.isin(session_to_exclude)].reset_index(drop=True)

with open(Path(processed_dir, "glm_hmm_models", "glm_hmm_masked_final.pkl"), "rb") as f:
    glm_hmm = pickle.load(f)
glm_hmm_original = copy.deepcopy(glm_hmm)

with open(Path(processed_dir, "ephys_neuron_wise.pkl"), "rb") as f:
    ephys = pickle.load(f)

## Extract trial info (blocks or glm-hmm states)

In [ ]:
data = glm_hmm["data"]

# HMM states — also flips sign for awayRF sessions in-place on glm_hmm["data"]
# compiled_dir loads reaction_time from trial CSVs (required by get_trial_num)
biased_state_trial_info, unbiased_state_trial_info, state_occupancy = \
    dpca_utils.extract_hmm_state_trial_info(session_metadata, glm_hmm_original, data,
                                            compiled_dir=compiled_dir)

# Blocks from prob_toRF (run after extract_hmm_state_trial_info so awayRF sign flip is applied)
equal_block_trial_info, unequal_block_trial_info = \
    dpca_utils.extract_block_trial_info(data, session_metadata["session_id"])

## Shared setup

In [ ]:
toRF_sessions  = session_metadata.session_id[session_metadata.prior_direction == "toRF"]
awayRF_sessions = session_metadata.session_id[session_metadata.prior_direction == "awayRF"]

alignments = list(ephys_config["alignment_settings_GP"].keys())  # ['baseline','visual','cue','response']
marginalization_keys = ['b', 's', 'c', 't']

condition_dict_states = {
    "state_values": ["biased", "unbiased"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

state_trial_info = {
    "biased":   biased_state_trial_info,
    "unbiased": unbiased_state_trial_info,
}

COH_LABELS = ["0%", "6%", "20%", "50%"]


---
## Example 3 — 10% neuron leave-out (repeated for stability)

Each repeat randomly drops 10% of neurons. Pass the same `rng` seed for
reproducibility, or a different seed per repeat for a stability analysis.

In [ ]:
N_REPEATS = 10
all_projections_leaveout = []

for repeat in range(N_REPEATS):
    neuron_ids_leaveout = dpca_utils.get_neuron_ids(
        neuron_metadata, toRF_sessions,
        leave_out_fraction=0.1,
        rng=np.random.default_rng(repeat),
    )

    avg, tw = dpca_utils.create_dpca_matrix(
        toRF_sessions, condition_dict_states, neuron_ids_leaveout,
        state_trial_info, neuron_metadata, ephys, ephys_config,
        condition_type="states",
    )
    fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

    results   = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
    projections = dpca_utils.cross_period_projection(results, fit_avg, alignments)
    all_projections_leaveout.append(projections)

In [ ]:
N_REPEATS = 10


for repeat in range(N_REPEATS):
    neuron_ids_leaveout = dpca_utils.get_neuron_ids(
        neuron_metadata, toRF_sessions,
        leave_out_fraction=0.2,
        rng=np.random.default_rng(repeat),
    )
    print(f"neurons not included in repeat {repeat}: {set(dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)) - set(neuron_ids_leaveout)}")

#### Build time axes from last repeat's `fit_avg`

In [ ]:
time_axes_lo = dpca_utils.build_time_axes(fit_avg, ephys_config, alignments)

#### Self-projection — 10 repeats separately

In [ ]:
PC = 0
margs_to_plot = ['b', 's', 'c']

BIASED_COLORS   = dpca_plot_utils.BIASED_COLORS
UNBIASED_COLORS = dpca_plot_utils.UNBIASED_COLORS
MARG_LABELS     = dpca_plot_utils.MARG_LABELS
ALIGN_LABELS    = dpca_plot_utils.ALIGN_LABELS

for marg in margs_to_plot:
    n_repeats = len(all_projections_leaveout)
    n_cols    = len(alignments)
    fig, axes = plt.subplots(n_repeats, n_cols,
                             figsize=(5 * n_cols, 3 * n_repeats),
                             sharex="col", sharey=False)
    for row, proj in enumerate(all_projections_leaveout):
        for col, alignment in enumerate(alignments):
            ax = axes[row, col]
            Z  = proj[alignment][alignment]
            dpca_plot_utils.plot_dpca_traces(ax, time_axes_lo[alignment], Z[marg][PC])
            ax.axvline(0, color="k", linestyle="--", linewidth=0.8)
            if row == 0:
                ax.set_title(ALIGN_LABELS.get(alignment, alignment), fontsize=12)
            if col == 0:
                ax.set_ylabel(f"Repeat {row + 1}", fontsize=10)
    fig.suptitle(
        f"Self-projection — {MARG_LABELS.get(marg, marg)} PC{PC + 1} — 10% leave-out",
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    plt.show()

#### Cross-projection (fit on baseline) — 10 repeats separately

In [ ]:
fit_align = "baseline"
PC = 0
margs_to_plot = ['b']

for marg in margs_to_plot:
    n_repeats = len(all_projections_leaveout)
    n_cols    = len(alignments)
    fig, axes = plt.subplots(n_repeats, n_cols,
                             figsize=(5 * n_cols, 3 * n_repeats),
                             sharex="col", sharey=False)
    for row, proj in enumerate(all_projections_leaveout):
        for col, proj_align in enumerate(alignments):
            ax = axes[row, col]
            Z  = proj[fit_align][proj_align]
            dpca_plot_utils.plot_dpca_traces(ax, time_axes_lo[proj_align], Z[marg][PC])
            if row == 0:
                ax.set_title(ALIGN_LABELS.get(proj_align, proj_align), fontsize=12)
            if col == 0:
                ax.set_ylabel(f"Repeat {row + 1}", fontsize=10)
            if proj_align == fit_align:
                for spine in ax.spines.values():
                    spine.set_edgecolor("#e05c00")
                    spine.set_linewidth(2)
    fig.suptitle(
        f"Cross-projection (fit={fit_align}) — {MARG_LABELS.get(marg, marg)} PC{PC + 1}"
        " — 10% leave-out",
        fontsize=13, y=1.01,
    )
    plt.tight_layout()
    plt.show()

#### Cross-projection stability — mean ± std across repeats (coherences collapsed)

In [ ]:
fit_align  = "baseline"
marg       = "b"
PC         = 0

# state × choice colors/linestyles (one entry per state, dashed = toRF choice)
state_colors = [BIASED_COLORS[-1], UNBIASED_COLORS[-1]]
ls_map       = ["-", "--"]
state_labels = ["biased", "unbiased"]
choice_labels = ["awayRF", "toRF"]

# collect coherence-averaged traces: (n_repeats, n_states, n_choices, n_time) per alignment
traces = {
    proj_align: np.stack([
        proj[fit_align][proj_align][marg][PC].mean(axis=1)   # mean over coh axis
        for proj in all_projections_leaveout
    ])
    for proj_align in alignments
}
# traces[proj_align] shape: (n_repeats, n_states, n_choices, n_time)

fig, axes = plt.subplots(1, len(alignments), figsize=(5 * len(alignments), 4), sharey=False)

for col, proj_align in enumerate(alignments):
    ax   = axes[col]
    t    = time_axes_lo[proj_align]
    data = traces[proj_align]          # (n_repeats, n_states, n_choices, n_time)
    mn   = data.mean(axis=0)           # (n_states, n_choices, n_time)
    sd   = data.std(axis=0)            # (n_states, n_choices, n_time)

    for b, (color, slabel) in enumerate(zip(state_colors, state_labels)):
        for c, (ls, clabel) in enumerate(zip(ls_map, choice_labels)):
            ax.plot(t, mn[b, c], color=color, linestyle=ls, linewidth=1.8,
                    label=f"{slabel} {clabel}" if col == 0 else "_nolegend_")
            ax.fill_between(t, mn[b, c] - sd[b, c], mn[b, c] + sd[b, c],
                            color=color, alpha=0.25)

    ax.axvline(0, color="k", linestyle="--", linewidth=0.8)
    ax.set_title(ALIGN_LABELS.get(proj_align, proj_align), fontsize=12)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if proj_align == fit_align:
        for spine in ax.spines.values():
            spine.set_edgecolor("#e05c00")
            spine.set_linewidth(2)

fig.legend(loc="upper right", fontsize=10, frameon=False)
fig.suptitle(
    f"Cross-projection (fit={fit_align}) — Bias PC{PC + 1}"
    f" — mean ± std across {len(all_projections_leaveout)} repeats (20% leave-out)",
    fontsize=12, y=1.02,
)
plt.tight_layout()
plt.show()
fig.savefig(Path("../dissemination/dpca/dpca_toRF_session_all_neuron") / "cross_projection_cv.png", dpi=150, bbox_inches="tight")

---
## Example 5 — Blocks instead of HMM states

Block membership comes from `glm_hmm["data"][session_id]["prob_toRF"]`:
- `prob_toRF == 50` → equal block
- `prob_toRF != 50` → unequal block

In [ ]:
condition_dict_blocks = {
    "state_values": ["equal", "unequal"],
    "coherences":   [0, 0.06, 0.2, 0.5],
    "choices":      ["awayRF", "toRF"],
}

block_trial_info = {
    "equal":   equal_block_trial_info,
    "unequal": unequal_block_trial_info,
}

neuron_ids = dpca_utils.get_neuron_ids(neuron_metadata, toRF_sessions)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_blocks = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_blocks  = dpca_utils.cross_period_projection(dpca_results_blocks, full_avg, alignments)

---
## Example 6 — Combining filters: exclude trash, 10% leave-out, blocks

In [ ]:
neuron_ids_combined = dpca_utils.get_neuron_ids(
    neuron_metadata, toRF_sessions,
    exclude_cell_types=["trash"],
    leave_out_fraction=0.1,
    rng=np.random.default_rng(0),
)

avg, tw = dpca_utils.create_dpca_matrix(
    toRF_sessions, condition_dict_blocks, neuron_ids_combined,
    block_trial_info, neuron_metadata, ephys, ephys_config,
    condition_type="blocks",
)
fit_avg, fit_tw, full_avg, full_tw = dpca_utils.clean_dpca_data(avg, tw, alignments)

dpca_results_combined = dpca_utils.fit_dpca_all_alignments(fit_avg, fit_tw, alignments)
projections_combined  = dpca_utils.cross_period_projection(dpca_results_combined, full_avg, alignments)